In [40]:
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.cuda as cuda
import time

In [2]:
# Setting up cuda
device = 'cuda' if cuda.is_available() else 'cpu'
print(f"The current device is: {device}")

The current device is: cuda


In [3]:
# Loading the dataset from Hugging Face
train_dataset = load_dataset("ylecun/mnist", split= 'train')
test_dataset = load_dataset("ylecun/mnist", split = 'test')

In [4]:
# Turn image into Torch tensor
train_dataset = train_dataset.with_format("torch")
test_dataset = test_dataset.with_format("torch")

In [5]:
# Collate function to turn dtype into torch expected dtype for forward() and loss
def collate_func(batch):
    image = torch.stack([item['image'].type(torch.float) for item in batch])
    label = torch.stack([item['label'].type(torch.uint8) for item in batch])
    return {'image':image, 'label':label}

In [6]:
# Turn data into Torch DataLoader object
train_dataloader = DataLoader(train_dataset,
                              batch_size=32,
                              shuffle=True,
                              collate_fn=collate_func)

test_dataloader = DataLoader(test_dataset,
                             batch_size=32,
                             shuffle=True,
                             collate_fn=collate_func)

In [7]:
# Building the model
class ImageClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=8,
                kernel_size=(3,3),
                padding=1,
                stride=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(3,3),
                        stride=3,
                        padding=1),
            nn.ReLU()
        )
        self.connected_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=8 * 10 * 10,
                      out_features=32),
            nn.ReLU(),
            nn.Linear(in_features=32,
                      out_features=10)
        )
    def forward(self, x : torch.Tensor):
        outputs = self.connected_block(self.convolution_block_1(x))
        return outputs

In [24]:
# Function to calculate prediction accuracy
def get_accuracy(logits, labels):
    predictions = torch.argmax(logits, dim = 1)
    correct_predictions = torch.sum(predictions == labels)
    accuracy = correct_predictions / len(labels) * 100
    return accuracy

In [38]:
# Create the train loop for the model
def train_step(model, data_loader, loss_fn, optimizer, device, accuracy_fn):
    model.train()
    loss = 0
    accuracy = 0
    for batch in data_loader:
        images, labels = batch['image'].to(device), batch['label'].to(device)
        
        outputs = model(images)
        batch_loss = loss_fn(outputs, labels)
        batch_accuracy = accuracy_fn(outputs, labels)

        loss += batch_loss.item()
        accuracy += batch_accuracy
        
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()
    loss /= len(data_loader)
    accuracy /= len(data_loader)
    return loss, accuracy

In [56]:
# Create the test function
def evaluate(model, data_loader, loss_fn, device, accuracy_fn):
    model.eval()
    loss = 0
    accuracy = 0
    with torch.inference_mode():
        for batch in data_loader:
            images, labels = batch['image'].to(device), batch['label'].to(device)
            outputs = model(images)
            
            batch_loss = loss_fn(outputs, labels)
            batch_accuracy = get_accuracy(outputs, labels)
            
            loss += batch_loss.item()
            accuracy += batch_accuracy
        loss /= len(data_loader)
        accuracy /= len(data_loader)
    return loss, accuracy

In [60]:
# Instantiate the model, optimizer and loss function
torch.manual_seed(42)
model = ImageClassificationModel()
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_function = nn.CrossEntropyLoss()

model.to(device)
loss_function.to(device)

CrossEntropyLoss()

In [61]:
# Training the model
epochs = 3
start_time = time.time()
for epoch in range(epochs):
    train_loss, train_accuracy = train_step(model=model,
                                data_loader=train_dataloader,
                                loss_fn=loss_function,
                                optimizer=optimizer,
                                device=device,
                                accuracy_fn=get_accuracy
                               )
end_time = time.time()
train_time = end_time - start_time

In [62]:
# Testing the model on test set
test_loss, test_accuracy = evaluate(model=model,
                                    data_loader=test_dataloader,
                                    loss_fn=loss_function,
                                    device = device,
                                    accuracy_fn = get_accuracy)

In [64]:
# Printing the metrics
print(f"Train time: {train_time} seconds")
print(f"Train loss: {train_loss}, train accuracy: {train_accuracy}")
print(f"Test loss: {test_loss}, test accuracy: {test_accuracy}")

Train time: 14.532257318496704 seconds
Train loss: 0.3594834790999691, train accuracy: 90.21666717529297
Test loss: 0.3606472575721649, test accuracy: 90.6150131225586
